# Tikhonov regularization with KL data fidelity term
We consider the two-dimensional deconvolution problems to find a non-negative function f given data 
$$
    d \sim \mathrm{Pois}(h*f)
$$
with a non-negative convolution kernel $h$, and $\mathrm{Pois}$ denotes the element-wise Poisson distribution.

We first study constrained quadratic Tikhonov regularization 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\mathrm{KL}(d+\sigma,h*f+\sigma) + \alpha \|f\|^2_{L^2}\right]
$$
with the Kullback-Leibler divergence as data fidelity term and an offset $\sigma>0$.
We also test entropy regularization given by 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\mathrm{KL}(d+\sigma,h*f+\sigma) + \alpha \mathrm{KL}(f,1)\right]
$$

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mplib

from regpy.operators.convolution import GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import TikhonovRegularizationSetting, RegularizationSetting
from regpy.solvers.linear.semismoothNewton import SemismoothNewton_nonneg
from regpy.solvers.linear.proximal_gradient import ForwardBackwardSplitting, FISTA
from regpy.solvers.linear.primal_dual import PDHG
from regpy.hilbert import L2
from regpy.stoprules import DualityGapStopping
from regpy.functionals import QuadraticLowerBound, QuadraticBilateralConstraints, KullbackLeibler, RelativeEntropy
from comparison_plot import comparison_plot

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

### test objects

In [ ]:
grid = UniformGridFcts((-1, 1, 256), (-1.5, 1, 256),dtype = float, periodic = True)
"""Space of real-valued functions on a uniform grid with rectangular pixels"""
X = grid.coords[0]; Y = grid.coords[1]
"""x and y coordinates."""
cross = 1.0*np.logical_or((abs(X)<0.01) * (abs(Y)<0.3),(abs(X)<0.3) * (abs(Y)<0.01)) 
rad = np.sqrt(X**2 + Y**2)
ring = 1.0*np.logical_and(rad>=0.9, rad<=0.95)
smallbox = (abs(X+0.55)<=0.05) * (abs(Y-0.55)<=0.05)
bubbles = (1.001+np.sin(50/(X+1.3)))*np.exp(-((Y+1.25)/0.1)**2)*(X>-0.8)*(X<0.8)

ramp = Y<=-1

objects = 200*(ring + 2.0*cross + 1.5*smallbox + 2*ramp -bubbles)
exact_sol = objects 


### creating Poisson distributed synthetic data

In [ ]:
a=0.15
conv =  GaussianBlur(grid,a,pad_amount=16)
r"""Convolution operator $f\mapsto h*f$ for the convolution kernel $h(x)=\exp(-|x|_2^2/a^2)$."""
blur = conv(exact_sol)
blur[blur<0] = 0.
"""Simulated exact data."""
data = np.random.poisson(blur)
"""Simulated measured data. The Poisson distribution occurs if photon count detectors are used."""
comparison_plot(grid,exact_sol,data,title_left='noisy measurement data')

## Kullback-Leibler data fidelity with nonnegativity-contrained $L^2$ penalty

In [ ]:
sigma = 1.
KL_shift = KullbackLeibler(grid,w=data+sigma).shift(np.broadcast_to(-sigma,grid.shape))
penLower = QuadraticLowerBound(grid,x0=0,lb=0)
alpha = 1e-3
n_iter = 400 
settingLower = TikhonovRegularizationSetting(op=conv, penalty=penLower, data_fid = KL_shift,regpar=alpha)


### Forward-backward splitting

In [ ]:
FB_solver_lb = ForwardBackwardSplitting(settingLower)
stop_FB_lb=DualityGapStopping(FB_solver_lb,threshold = 1., max_iter=n_iter,logging_level=logging.WARNING)
FB_solver_lb.run(stoprule=stop_FB_lb)

comparison_plot(grid,exact_sol,FB_solver_lb.x,title_left='Forward-backward')

### FISTA

In [ ]:
FISTA_solver_lb = FISTA(settingLower)
n_iter= 1000
stop_FISTA_lb=DualityGapStopping(FISTA_solver_lb,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
FISTA_solver_lb.run(stoprule=stop_FISTA_lb)
    
comparison_plot(grid,exact_sol,FISTA_solver_lb.x,title_left='FISTA reco')

### Primal-dual hybrid gradient (PDHG) method

In [ ]:
PDHG_solver_lb = PDHG(settingLower)
stop_PDHG_lb=DualityGapStopping(PDHG_solver_lb,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_solver_lb.run(stoprule=stop_PDHG_lb)

comparison_plot(grid,exact_sol,PDHG_solver_lb.x, title_left='PDHG reco')

### comparison of convergence speeds

In [ ]:
plt.semilogy(stop_FB_lb.gap_stat,label='ForwardBackward')
plt.semilogy(stop_FISTA_lb.gap_stat,label='FISTA')
plt.semilogy(stop_PDHG_lb.gap_stat,label='PDHG')
plt.legend()
plt.xlabel('it. step'); plt.ylabel('duality gap')
plt.title('convergence for nonnegativity constraint')

## Kullback-Leibler data fidelity with relative entropy penalty

In [ ]:
RE = RelativeEntropy(grid,w =grid.ones())
alpha = 1e-3
n_iter = 400 
settingRE = TikhonovRegularizationSetting(op=conv, penalty=penLower, data_fid = KL_shift,regpar=alpha)

### Forward-backward splitting

In [ ]:
FB_solver_RE = ForwardBackwardSplitting(settingRE)
stop_FB_RE=DualityGapStopping(FB_solver_RE,threshold = 1., max_iter=n_iter,logging_level=logging.WARNING)
FB_solver_RE.run(stoprule=stop_FB_RE)

comparison_plot(grid,exact_sol,FB_solver_RE.x,title_left='Forward-backward rel. entropy')

### FISTA

In [ ]:
FISTA_solver_RE = FISTA(settingRE)
stop_Fista_RE=DualityGapStopping(FISTA_solver_RE,threshold = 1., max_iter=n_iter,logging_level=logging.WARNING)
FISTA_solver_RE.run(stoprule=stop_Fista_RE)

comparison_plot(grid,exact_sol,FISTA_solver_RE.x,title_left='FISTA rel. entropy')


### Primal-dual hybrid gradient (PDHG) method

In [ ]:
PDHG_solver_RE = PDHG(settingRE)
stop_PDHG_RE=DualityGapStopping(PDHG_solver_RE,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_solver_RE.run(stoprule=stop_PDHG_RE)

comparison_plot(grid,exact_sol,PDHG_solver_RE.x, title_left='PDHG reco')

### comparison of convergence speeds

In [ ]:
plt.semilogy(stop_FB_RE.gap_stat,label='ForwardBackward')
plt.semilogy(stop_Fista_RE.gap_stat,label='FISTA')
plt.semilogy(stop_PDHG_RE.gap_stat,label='PDHG')
plt.legend()
plt.xlabel('it. step'); plt.ylabel('duality gap')
plt.title('convergence for relative entropy regularization')